# Lab 4.4: MCP Server Integration

**What you'll build:** MCP server configurations for team and personal use -- and learn how to manage secrets without committing them.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- hardcoded secrets in MCP config | 5 min |
| 2 | The Right Way -- environment variable expansion and proper scoping | 5 min |
| 3 | Your Turn -- configure MCP servers for a realistic project | 10 min |
| 4 | Validation -- verify your configs are correct and secure | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

MCP (Model Context Protocol) servers extend Claude's capabilities with custom tools. You configure them in two places:

| Config File | Location | Scope | Use For |
|------------|---------|-------|--------|
| `.mcp.json` | Project root | Shared (version controlled) | Team tools |
| `~/.claude.json` | Home directory | Personal (not shared) | Your own experimental tools |

Both are discovered at connection time. Tools from both configs are available simultaneously.

The key challenge: `.mcp.json` is committed to your repo. How do you handle secrets?

---

## Phase 1: The Wrong Approach

A developer commits an MCP config with a hardcoded API key.

In [ ]:
# BAD: Hardcoded secrets in .mcp.json
BAD_CONFIG = {
    "mcpServers": {
        "jira": {
            "command": "npx",
            "args": ["-y", "@mcp/jira-server"],
            "env": {
                "JIRA_URL": "https://mycompany.atlassian.net",
                "JIRA_TOKEN": "ATT-real-api-token-12345abcdef"  # <-- SECRET IN CODE!
            }
        },
        "github": {
            "command": "npx",
            "args": ["-y", "@mcp/github-server"],
            "env": {
                "GITHUB_TOKEN": "ghp_realtoken789xyz"  # <-- SECRET IN CODE!
            }
        }
    }
}


def audit_mcp_config(config: dict) -> list[str]:
    """Audit an MCP config for security issues."""
    issues = []
    for server_name, server_config in config.get("mcpServers", {}).items():
        env = server_config.get("env", {})
        for key, value in env.items():
            if isinstance(value, str) and not value.startswith("${"):
                # Check if it looks like a secret
                secret_patterns = ["token", "key", "secret", "password", "credential"]
                if any(pattern in key.lower() for pattern in secret_patterns):
                    issues.append(
                        f"SECURITY: '{server_name}' has hardcoded secret in '{key}'. "
                        f"Use ${{{key}}} environment variable expansion instead."
                    )
    return issues


print("=== SECURITY AUDIT: BAD CONFIG ===")
issues = audit_mcp_config(BAD_CONFIG)
for issue in issues:
    print(f"  [FAIL] {issue}")
print(f"\nFound {len(issues)} security issue(s).")
print("\nThese secrets would be committed to version control!")

---

## Phase 2: The Right Approach

Use `${ENV_VAR}` syntax to reference secrets from environment variables.

In [ ]:
# GOOD: Environment variable expansion for secrets
GOOD_PROJECT_CONFIG = {
    "mcpServers": {
        "jira": {
            "command": "npx",
            "args": ["-y", "@mcp/jira-server"],
            "env": {
                "JIRA_URL": "https://mycompany.atlassian.net",  # Non-secret: OK to hardcode
                "JIRA_TOKEN": "${JIRA_TOKEN}"  # Secret: uses env var expansion
            }
        },
        "github": {
            "command": "npx",
            "args": ["-y", "@mcp/github-server"],
            "env": {
                "GITHUB_TOKEN": "${GITHUB_TOKEN}"  # Secret: uses env var expansion
            }
        },
        "filesystem": {
            "command": "npx",
            "args": ["-y", "@mcp/filesystem-server", "/project/docs"],
            "env": {}  # No secrets needed for local filesystem
        }
    }
}

# Personal config -- experimental servers
GOOD_PERSONAL_CONFIG = {
    "mcpServers": {
        "my-custom-db": {
            "command": "python",
            "args": ["/home/dev/mcp-servers/db-server.py"],
            "env": {
                "DB_CONNECTION": "${MY_DEV_DB_URL}"  # Personal DB, not shared
            }
        }
    }
}

print("=== SECURITY AUDIT: GOOD PROJECT CONFIG (.mcp.json) ===")
issues = audit_mcp_config(GOOD_PROJECT_CONFIG)
if not issues:
    print("  [PASS] No hardcoded secrets found.")
    print("  Safe to commit to version control.")
else:
    for issue in issues:
        print(f"  [FAIL] {issue}")

print(f"\n=== PERSONAL CONFIG (~/.claude.json) ===")
print("  This file is NOT version controlled.")
print("  Contains personal/experimental server configs.")
print(f"  Servers: {list(GOOD_PERSONAL_CONFIG['mcpServers'].keys())}")

# Show what's available at connection time
all_servers = set(GOOD_PROJECT_CONFIG["mcpServers"].keys()) | set(GOOD_PERSONAL_CONFIG["mcpServers"].keys())
print(f"\n=== AT CONNECTION TIME ===")
print(f"  Total servers available: {len(all_servers)}")
print(f"  From .mcp.json (team): {list(GOOD_PROJECT_CONFIG['mcpServers'].keys())}")
print(f"  From ~/.claude.json (personal): {list(GOOD_PERSONAL_CONFIG['mcpServers'].keys())}")
print(f"  All available: {sorted(all_servers)}")

---

## Phase 3: Your Turn

Configure MCP servers for a realistic project scenario.

**Scenario:** You're on a team building a data analytics platform. You need:
1. A **Postgres MCP server** for database queries (team tool, needs DB credentials)
2. A **Slack MCP server** for notifications (team tool, needs Slack token)
3. A **custom visualization server** you're experimenting with (personal, not ready for team)

**Your goal:** Create both `.mcp.json` (team) and `~/.claude.json` (personal) configs with proper secret management.

In [ ]:
# =============================================================
# TODO: Create your MCP configurations
# =============================================================
#
# Requirements:
# - Team config (.mcp.json): postgres + slack servers
#   - Postgres needs: POSTGRES_URL (secret), POSTGRES_DB (not secret)
#   - Slack needs: SLACK_TOKEN (secret), SLACK_CHANNEL (not secret)
#   - Use ${ENV_VAR} for secrets, hardcode non-secrets
#
# - Personal config (~/.claude.json): visualization server
#   - Command: python, args: path to your local viz-server.py
#   - Needs VIZ_API_KEY (your personal key)

YOUR_PROJECT_CONFIG = {
    "mcpServers": {
        # TODO: Add postgres server config
        # TODO: Add slack server config
    }
}

YOUR_PERSONAL_CONFIG = {
    "mcpServers": {
        # TODO: Add visualization server config
    }
}

# Validate
print("=== VALIDATING YOUR CONFIGS ===\n")

# Check project config
project_servers = YOUR_PROJECT_CONFIG.get("mcpServers", {})
checks = []

# Postgres checks
if "postgres" not in project_servers:
    checks.append(("FAIL", "Missing 'postgres' server in project config"))
else:
    pg = project_servers["postgres"]
    pg_env = pg.get("env", {})
    if pg_env.get("POSTGRES_URL", "").startswith("${"):
        checks.append(("PASS", "Postgres URL uses env var expansion"))
    elif "POSTGRES_URL" in pg_env:
        checks.append(("FAIL", "Postgres URL should use ${POSTGRES_URL}"))
    else:
        checks.append(("FAIL", "Missing POSTGRES_URL in postgres env"))

# Slack checks
if "slack" not in project_servers:
    checks.append(("FAIL", "Missing 'slack' server in project config"))
else:
    sl = project_servers["slack"]
    sl_env = sl.get("env", {})
    if sl_env.get("SLACK_TOKEN", "").startswith("${"):
        checks.append(("PASS", "Slack token uses env var expansion"))
    elif "SLACK_TOKEN" in sl_env:
        checks.append(("FAIL", "Slack token should use ${SLACK_TOKEN}"))
    else:
        checks.append(("FAIL", "Missing SLACK_TOKEN in slack env"))

# Security audit
security_issues = audit_mcp_config(YOUR_PROJECT_CONFIG)
if security_issues:
    for issue in security_issues:
        checks.append(("FAIL", issue))
else:
    checks.append(("PASS", "No hardcoded secrets in project config"))

# Personal config checks
personal_servers = YOUR_PERSONAL_CONFIG.get("mcpServers", {})
if len(personal_servers) == 0:
    checks.append(("FAIL", "Missing visualization server in personal config"))
else:
    checks.append(("PASS", f"Personal config has {len(personal_servers)} server(s)"))

for status, msg in checks:
    print(f"  [{status}] {msg}")

passed = sum(1 for s, _ in checks if s == "PASS")
total = len(checks)
print(f"\nScore: {passed}/{total}")
if passed == total:
    print("PASSED -- your MCP configs are correctly structured and secure!")

---

## Phase 4: Config Review Exercise

Use Claude to review MCP configurations for security and best practices.

In [ ]:
# Ask Claude to review an MCP config
REVIEW_CONFIG = json.dumps({
    "mcpServers": {
        "database": {
            "command": "npx",
            "args": ["-y", "@mcp/postgres-server"],
            "env": {
                "DB_HOST": "prod-db.company.com",
                "DB_PASSWORD": "SuperSecret123!",
                "DB_NAME": "analytics"
            }
        },
        "internal-api": {
            "command": "node",
            "args": ["./mcp-servers/api-server.js"],
            "env": {
                "API_ENDPOINT": "https://api.company.com/v2",
                "API_KEY": "${COMPANY_API_KEY}"
            }
        }
    }
}, indent=2)

review_prompt = f"""Review this .mcp.json configuration for security issues and best practices.
This file will be committed to a shared Git repository.

```json
{REVIEW_CONFIG}
```

For each server, check:
1. Are any secrets hardcoded instead of using ${{ENV_VAR}} expansion?
2. Is the command/args configuration reasonable?
3. Any other security concerns?

Return your findings as a JSON array with objects containing:
- "server": server name
- "issue": description of the problem
- "fix": how to fix it
- "severity": "critical", "warning", or "info"

If a server has no issues, include it with issue: "No issues found" and severity: "info".
"""

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": review_prompt}],
)

raw = response.content[0].text.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

try:
    findings = json.loads(raw)
    print("Claude's MCP Config Review:\n")
    for f in findings:
        sev = f.get("severity", "info").upper()
        print(f"  [{sev}] {f.get('server', '?')}")
        print(f"    Issue: {f.get('issue', '?')}")
        print(f"    Fix: {f.get('fix', '?')}")
        print()
except json.JSONDecodeError:
    print(f"Raw response:\n{raw}")

### Key Takeaways

1. **`.mcp.json` = team tools** (version controlled, shared). **`~/.claude.json` = personal tools** (not shared).
2. **Use `${ENV_VAR}` for secrets** in `.mcp.json`. Never commit API keys, tokens, or passwords.
3. **Both configs are active simultaneously.** They merge at connection time -- you get all servers from both.
4. **Non-secret values can be hardcoded.** URLs, database names, channel names are fine. Only credentials need environment variable expansion.
5. **Community servers need due diligence.** Review source code, check tool description quality, verify secret handling before adopting.